# BPE Tokenizer from Scratch

Byte-Pair Encoding: the dominant subword tokenization algorithm used in GPT, LLaMA, and most modern LLMs.

**Interview questions:**
- "How does BPE work?"
- "Why subword tokenization over word-level?"
- "BPE vs WordPiece vs Unigram — what are the differences?"

---

## Why Subword Tokenization?

| Method | Vocabulary | OOV Problem | Efficiency |
|--------|-----------|-------------|------------|
| Character-level | ~256 | None | Very long sequences |
| Word-level | 50K-500K | Severe | Short sequences |
| **Subword (BPE)** | 32K-128K | **None** (fallback to chars) | **Good balance** |

Subword tokenization gives the best of both worlds:
- Common words are single tokens ("the", "and")
- Rare/new words decompose into known pieces ("unhappiness" -> "un" + "happiness")
- Never OOV (can always fall back to byte-level)

In [ ]:
import re
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import numpy as np

## 1. BPE Algorithm

**Training procedure:**
1. Start with a vocabulary of individual characters (or bytes)
2. Count all adjacent pairs in the training corpus
3. Merge the most frequent pair into a new token
4. Repeat steps 2-3 until desired vocabulary size

**Encoding (inference):** apply learned merges in order to segment new text.

In [ ]:
class BPETokenizer:
    """Byte-Pair Encoding tokenizer from scratch.
    
    This implementation uses the word-level BPE approach (as in the original paper):
    - Pre-tokenize into words
    - Represent each word as a sequence of characters + end-of-word marker
    - Learn merges across the corpus
    """
    def __init__(self):
        self.merges = []        # List of (pair, merged_token) in order
        self.vocab = {}         # token -> id
        self.inverse_vocab = {} # id -> token
        self.special_tokens = {}
    
    def _pre_tokenize(self, text: str) -> list[str]:
        """Split text into words (whitespace + punctuation boundaries).
        GPT-2 uses a regex pattern; we use a simplified version.
        """
        # GPT-2 style: split on whitespace, keep the space as prefix of next word
        pattern = r"""'s|'t|'re|'ve|'m|'ll|'d| ?\w+| ?[^\s\w]+|\s+(?!\S)|\s+"""
        return re.findall(pattern, text)
    
    def _get_word_freqs(self, corpus: str) -> dict[tuple, int]:
        """Get frequency of each word (as tuple of characters)."""
        words = self._pre_tokenize(corpus)
        word_freqs = Counter()
        for word in words:
            # Convert word to tuple of chars
            chars = tuple(word)
            word_freqs[chars] += 1
        return dict(word_freqs)
    
    def _count_pairs(self, word_freqs: dict) -> Counter:
        """Count frequency of all adjacent pairs across the corpus."""
        pairs = Counter()
        for word, freq in word_freqs.items():
            for i in range(len(word) - 1):
                pairs[(word[i], word[i+1])] += freq
        return pairs
    
    def _merge_pair(self, word_freqs: dict, pair: tuple) -> dict:
        """Merge all occurrences of pair in word_freqs."""
        new_word_freqs = {}
        a, b = pair
        merged = a + b
        
        for word, freq in word_freqs.items():
            new_word = []
            i = 0
            while i < len(word):
                if i < len(word) - 1 and word[i] == a and word[i+1] == b:
                    new_word.append(merged)
                    i += 2
                else:
                    new_word.append(word[i])
                    i += 1
            new_word_freqs[tuple(new_word)] = freq
        
        return new_word_freqs
    
    def train(self, corpus: str, vocab_size: int = 300, 
              special_tokens: list[str] = None, verbose: bool = True):
        """Train BPE on a corpus."""
        if special_tokens is None:
            special_tokens = ['<PAD>', '<UNK>', '<BOS>', '<EOS>']
        
        # Step 1: Initial vocabulary = unique characters
        word_freqs = self._get_word_freqs(corpus)
        
        # Collect initial character vocabulary
        base_vocab = set()
        for word in word_freqs:
            for char in word:
                base_vocab.add(char)
        base_vocab = sorted(base_vocab)
        
        # Build initial vocab: special tokens + characters
        self.vocab = {}
        for tok in special_tokens:
            self.vocab[tok] = len(self.vocab)
            self.special_tokens[tok] = self.vocab[tok]
        for char in base_vocab:
            self.vocab[char] = len(self.vocab)
        
        if verbose:
            print(f"Initial vocab size: {len(self.vocab)} ({len(special_tokens)} special + {len(base_vocab)} chars)")
            print(f"Target vocab size: {vocab_size}")
        
        # Step 2: Iteratively merge most frequent pairs
        self.merges = []
        merge_history = []
        num_merges = vocab_size - len(self.vocab)
        
        for i in range(num_merges):
            pairs = self._count_pairs(word_freqs)
            if not pairs:
                break
            
            best_pair = max(pairs, key=pairs.get)
            best_count = pairs[best_pair]
            
            # Merge
            word_freqs = self._merge_pair(word_freqs, best_pair)
            merged_token = best_pair[0] + best_pair[1]
            self.merges.append(best_pair)
            self.vocab[merged_token] = len(self.vocab)
            
            merge_history.append({
                'step': i, 'pair': best_pair, 'merged': merged_token,
                'count': best_count, 'vocab_size': len(self.vocab)
            })
            
            if verbose and (i < 10 or (i + 1) % 50 == 0):
                print(f"  Merge {i+1}: {best_pair!r} -> '{merged_token}' (count={best_count})")
        
        self.inverse_vocab = {v: k for k, v in self.vocab.items()}
        
        if verbose:
            print(f"\nFinal vocab size: {len(self.vocab)}")
            print(f"Total merges: {len(self.merges)}")
        
        return merge_history
    
    def encode(self, text: str) -> list[int]:
        """Encode text to token IDs."""
        words = self._pre_tokenize(text)
        token_ids = []
        
        for word in words:
            # Start with individual characters
            tokens = list(word)
            
            # Apply merges in order
            for a, b in self.merges:
                new_tokens = []
                i = 0
                while i < len(tokens):
                    if i < len(tokens) - 1 and tokens[i] == a and tokens[i+1] == b:
                        new_tokens.append(a + b)
                        i += 2
                    else:
                        new_tokens.append(tokens[i])
                        i += 1
                tokens = new_tokens
            
            # Convert tokens to IDs
            for tok in tokens:
                if tok in self.vocab:
                    token_ids.append(self.vocab[tok])
                else:
                    token_ids.append(self.special_tokens.get('<UNK>', 0))
        
        return token_ids
    
    def decode(self, token_ids: list[int]) -> str:
        """Decode token IDs back to text."""
        tokens = [self.inverse_vocab.get(id, '<UNK>') for id in token_ids]
        return ''.join(tokens)
    
    def tokenize(self, text: str) -> list[str]:
        """Return token strings (not IDs) for visualization."""
        words = self._pre_tokenize(text)
        all_tokens = []
        
        for word in words:
            tokens = list(word)
            for a, b in self.merges:
                new_tokens = []
                i = 0
                while i < len(tokens):
                    if i < len(tokens) - 1 and tokens[i] == a and tokens[i+1] == b:
                        new_tokens.append(a + b)
                        i += 2
                    else:
                        new_tokens.append(tokens[i])
                        i += 1
                tokens = new_tokens
            all_tokens.extend(tokens)
        
        return all_tokens

## 2. Train the Tokenizer

In [ ]:
# Training corpus
corpus = """
The quick brown fox jumps over the lazy dog. A quick brown cat leaps over the sleepy dog.
The transformer architecture revolutionized natural language processing. Transformers use
self-attention mechanisms to process sequences in parallel. The attention mechanism computes
weighted sums of value vectors based on query-key similarity. Multi-head attention allows
the model to attend to information from different representation subspaces. The feed-forward
network applies a non-linear transformation independently to each position. Layer normalization
stabilizes training by normalizing the activations. Residual connections help gradient flow
through deep networks. The encoder processes the input sequence bidirectionally while the
decoder generates output tokens autoregressively. Positional encodings provide position
information since attention is permutation equivariant. Pre-training on large corpora followed
by fine-tuning on downstream tasks is the standard paradigm. Large language models demonstrate
emergent abilities at sufficient scale. Scaling laws predict model performance as a function
of compute and data. Instruction tuning and RLHF align models with human preferences.
The tokenizer converts text into subword tokens that the model processes. Byte-pair encoding
is the most common tokenization algorithm for modern language models.
"""

tokenizer = BPETokenizer()
history = tokenizer.train(corpus, vocab_size=200, verbose=True)

In [ ]:
# Visualize vocabulary evolution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Merge frequency over time
counts = [h['count'] for h in history]
axes[0].plot(counts, 'b-', alpha=0.7)
axes[0].set_xlabel('Merge step')
axes[0].set_ylabel('Pair frequency')
axes[0].set_title('Most Frequent Pair Count at Each Step')
axes[0].grid(True, alpha=0.3)

# Token length distribution in final vocab
token_lengths = [len(tok) for tok in tokenizer.vocab if tok not in tokenizer.special_tokens]
axes[1].hist(token_lengths, bins=range(1, max(token_lengths)+2), 
             color='steelblue', edgecolor='white', align='left')
axes[1].set_xlabel('Token length (characters)')
axes[1].set_ylabel('Count')
axes[1].set_title('Token Length Distribution in Vocabulary')

plt.tight_layout()
plt.show()

## 3. Encoding and Decoding

In [ ]:
# Test encoding/decoding
test_texts = [
    "The quick brown fox",
    "attention mechanism",
    "transformer architecture",
    "an unseen word like xylophone",  # test unknown handling
]

for text in test_texts:
    tokens = tokenizer.tokenize(text)
    ids = tokenizer.encode(text)
    decoded = tokenizer.decode(ids)
    
    print(f"Text:    '{text}'")
    print(f"Tokens:  {tokens}")
    print(f"IDs:     {ids}")
    print(f"Decoded: '{decoded}'")
    print(f"Roundtrip match: {text == decoded}")
    print(f"Compression: {len(text)} chars -> {len(ids)} tokens ({len(text)/len(ids):.1f}x)")
    print()

## 4. Visualize Tokenization

In [ ]:
def visualize_tokenization(text, tokenizer):
    """Show tokenization with color-coded tokens."""
    tokens = tokenizer.tokenize(text)
    colors = plt.cm.Set3(np.linspace(0, 1, len(tokens)))
    
    fig, ax = plt.subplots(figsize=(14, 1.5))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    
    x_pos = 0.02
    for i, token in enumerate(tokens):
        display = token.replace(' ', '\u2423')  # show spaces
        width = len(display) * 0.018 + 0.02
        
        if x_pos + width > 0.98:
            break
        
        rect = plt.Rectangle((x_pos, 0.2), width, 0.6, 
                             facecolor=colors[i % len(colors)], 
                             edgecolor='gray', linewidth=0.5)
        ax.add_patch(rect)
        ax.text(x_pos + width/2, 0.5, display, ha='center', va='center', fontsize=9)
        x_pos += width + 0.005
    
    ax.set_title(f"Tokenization: {len(tokens)} tokens", fontsize=11)
    plt.tight_layout()
    plt.show()

visualize_tokenization("The transformer model uses attention mechanisms", tokenizer)
visualize_tokenization("natural language processing and deep learning", tokenizer)

## 5. Vocabulary Analysis

In [ ]:
# Show the longest (most merged) tokens
tokens_by_length = sorted(
    [(tok, len(tok)) for tok in tokenizer.vocab if tok not in tokenizer.special_tokens],
    key=lambda x: -x[1]
)

print("Longest tokens in vocabulary (most merged):")
for tok, length in tokens_by_length[:20]:
    print(f"  '{tok}' (length {length}, id={tokenizer.vocab[tok]})")

print(f"\nSingle-character tokens: {sum(1 for t, l in tokens_by_length if l == 1)}")
print(f"Multi-character tokens: {sum(1 for t, l in tokens_by_length if l > 1)}")

## 6. Byte-Level BPE (GPT-2 Style)

Real-world BPE operates on **bytes** (0-255) rather than Unicode characters. This:
- Handles any Unicode text without UNK tokens
- Base vocabulary is always exactly 256
- Used in GPT-2, GPT-3, GPT-4, LLaMA

In [ ]:
class ByteLevelBPE:
    """Byte-level BPE (simplified GPT-2 style).
    
    Operates on raw bytes, so any text can be encoded.
    Base vocab = 256 byte values.
    """
    def __init__(self):
        self.merges = []
        # Base vocab: single bytes (0-255)
        self.vocab = {bytes([i]): i for i in range(256)}
        self.inverse_vocab = {i: bytes([i]) for i in range(256)}
    
    def train(self, corpus: str, num_merges: int = 200, verbose: bool = True):
        """Train on corpus."""
        # Convert to bytes
        data = corpus.encode('utf-8')
        # Work with list of byte sequences (each word)
        words = corpus.split()  # simple splitting
        
        # Represent each word as a list of single bytes
        word_freqs = {}
        for word in words:
            word_bytes = tuple(bytes([b]) for b in word.encode('utf-8'))
            word_freqs[word_bytes] = word_freqs.get(word_bytes, 0) + 1
        
        for i in range(num_merges):
            # Count pairs
            pairs = Counter()
            for word, freq in word_freqs.items():
                for j in range(len(word) - 1):
                    pairs[(word[j], word[j+1])] += freq
            
            if not pairs:
                break
            
            best_pair = max(pairs, key=pairs.get)
            merged = best_pair[0] + best_pair[1]  # concatenate bytes
            
            # Merge in all words
            new_word_freqs = {}
            for word, freq in word_freqs.items():
                new_word = []
                j = 0
                while j < len(word):
                    if j < len(word) - 1 and word[j] == best_pair[0] and word[j+1] == best_pair[1]:
                        new_word.append(merged)
                        j += 2
                    else:
                        new_word.append(word[j])
                        j += 1
                new_word_freqs[tuple(new_word)] = freq
            word_freqs = new_word_freqs
            
            self.merges.append(best_pair)
            idx = len(self.vocab)
            self.vocab[merged] = idx
            self.inverse_vocab[idx] = merged
            
            if verbose and (i < 5 or (i + 1) % 50 == 0):
                print(f"  Merge {i+1}: {best_pair[0]!r} + {best_pair[1]!r} -> {merged!r} (count={pairs[best_pair]})")
        
        if verbose:
            print(f"Final vocab size: {len(self.vocab)}")
    
    def encode(self, text: str) -> list[int]:
        """Encode text to token IDs."""
        words = text.split()
        all_ids = []
        
        for word in words:
            tokens = [bytes([b]) for b in word.encode('utf-8')]
            
            for pair in self.merges:
                new_tokens = []
                i = 0
                while i < len(tokens):
                    if i < len(tokens) - 1 and tokens[i] == pair[0] and tokens[i+1] == pair[1]:
                        new_tokens.append(pair[0] + pair[1])
                        i += 2
                    else:
                        new_tokens.append(tokens[i])
                        i += 1
                tokens = new_tokens
            
            for tok in tokens:
                all_ids.append(self.vocab.get(tok, 0))
        
        return all_ids
    
    def decode(self, ids: list[int]) -> str:
        """Decode token IDs back to text."""
        byte_seq = b''.join(self.inverse_vocab[i] for i in ids)
        return byte_seq.decode('utf-8', errors='replace')

# Train byte-level BPE
byte_bpe = ByteLevelBPE()
byte_bpe.train(corpus, num_merges=100)

# Test
test = "transformer attention"
ids = byte_bpe.encode(test)
decoded = byte_bpe.decode(ids)
print(f"\nEncode '{test}' -> {ids}")
print(f"Decode -> '{decoded}'")

## 7. Comparison: BPE vs WordPiece vs Unigram

| | BPE | WordPiece | Unigram |
|---|-----|-----------|----------|
| **Training** | Bottom-up: merge frequent pairs | Bottom-up: merge by likelihood gain | Top-down: remove tokens by likelihood loss |
| **Encoding** | Apply merges in fixed order | Greedy longest-match | Most probable segmentation (Viterbi) |
| **Deterministic?** | Yes (fixed merge order) | Yes (greedy) | Can be probabilistic |
| **Used in** | GPT-2/3/4, LLaMA, Mistral | BERT, DistilBERT | T5, ALBERT, XLNet |
| **Subword prefix** | None (GPT) or ## | ## ("##ing") | \u2581 ("\u2581the") |

In [ ]:
# Demonstrate the conceptual difference
print("=" * 60)
print("Conceptual comparison on 'unhappiness':")
print("=" * 60)
print()
print("BPE approach (bottom-up merging):")
print("  Start: [u, n, h, a, p, p, i, n, e, s, s]")
print("  After merges: [un, happ, iness] (depends on corpus statistics)")
print()
print("WordPiece approach (greedy longest match):")
print("  Tries: 'unhappiness' -> not in vocab")
print("  Tries: 'unhappines' -> not in vocab")
print("  ...")
print("  Result: [un, ##happi, ##ness]")
print()
print("Unigram approach (probabilistic):")
print("  Finds most probable segmentation:")
print("  P('un' + 'happiness') vs P('unhapp' + 'iness') vs ...")
print("  Picks the highest probability segmentation")

## 8. Compression Analysis

In [ ]:
# Train tokenizers with different vocab sizes and measure compression
vocab_sizes = [50, 100, 150, 200, 250, 300]
compressions = []

test_text = "The transformer architecture uses self-attention mechanisms to process sequences"

for vs in vocab_sizes:
    tok = BPETokenizer()
    tok.train(corpus, vocab_size=vs, verbose=False)
    ids = tok.encode(test_text)
    ratio = len(test_text) / len(ids)
    compressions.append(ratio)

plt.figure(figsize=(8, 4))
plt.plot(vocab_sizes, compressions, 'bo-')
plt.xlabel('Vocabulary Size')
plt.ylabel('Compression Ratio (chars/token)')
plt.title('BPE Compression vs Vocabulary Size')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Larger vocab = higher compression (fewer tokens) but larger embedding matrix.")
print("Typical LLM vocab sizes: GPT-2=50257, LLaMA=32000, GPT-4~100000")

## Interview Notes

**"How does BPE work?"**
1. Start with character (or byte) vocabulary
2. Count all adjacent pair frequencies in corpus
3. Merge the most frequent pair into a new token
4. Repeat until desired vocab size
5. At inference: apply learned merges in order to segment text

**"Why subword over word-level?"**
- Word-level: fixed vocabulary, any new word is UNK
- Subword: can represent any word by decomposing into known pieces
- Handles morphology naturally: "unhappiness" = "un" + "happiness"
- Byte-level BPE guarantees zero UNK (always fallback to raw bytes)
- Balances vocabulary size with sequence length

**"Vocab size tradeoff?"**
- Small vocab (e.g., 8K): more tokens per text, longer sequences, more compute
- Large vocab (e.g., 128K): fewer tokens but larger embedding table, more memory
- Sweet spot for English: 32K-100K
- Multilingual models need larger vocabs (100K+)

**"Known issues with BPE?"**
- Tokenization of numbers is inconsistent ("123" might be ["1", "23"] or ["12", "3"])
- Whitespace handling varies by implementation
- Same word in different contexts may tokenize differently
- Not linguistically motivated — purely statistical